# Module 02 — Lesson 1: Prompt Evaluation & Graders with Claude

Welcome to the foundational lesson of **Prompt Engineering & Evaluation** from Anthropic's *Building with the Claude API*.

---

## 🎯 Why "Evals-First"?
Most engineers develop prompts by trial and error: changing a word, running a single test, and "eyeballing" the output. This approach fails in production because:
1. **Hidden regressions**: Improving one edge case often breaks another.
2. **Subjective bias**: What "feels" better on one run might score worse statistically across 100 runs.
3. **Zero reproducibility**: You cannot measure whether a smaller, faster model (e.g. Haiku) performs adequately compared to a larger one (Sonnet/Opus).

In this lesson, we build a **production-grade evaluation harness** *before* optimizing prompt wording.

---

## 🔄 The Evaluation Pipeline Architecture

```
                    ┌────────────────────────────┐
                    │ Synthetic Dataset Generator │
                    │ (Task + Format + Criteria) │
                    └─────────────┬──────────────┘
                                  │
                                  ▼
                    ┌────────────────────────────┐
                    │      Candidate Prompt      │
                    │   run_prompt(test_case)    │
                    └─────────────┬──────────────┘
                                  │
                   ┌──────────────┴──────────────┐
                   ▼                             ▼
       ┌───────────────────────┐     ┌───────────────────────┐
       │   Code-Based Grader   │     │   Model-Based Grader  │
       │ (ast, json, re parser)│     │   (LLM-as-a-Judge)    │
       │     Score: 0 or 10    │     │      Score: 1 - 10    │
       └───────────┬───────────┘     └───────────┬───────────┘
                   │                             │
                   └──────────────┬──────────────┘
                                  ▼
                    ┌────────────────────────────┐
                    │   Hybrid Composite Score   │
                    │  (model + syntax) / 2.0    │
                    └────────────────────────────┘
```

## Step 1: Client Setup & Anthropic Helper Functions

We initialize the `Anthropic` client and establish reusable message helpers:
- `add_user_message`: Appends a user turn to the conversation history.
- `add_assistant_message`: Appends an assistant turn (essential for **prefilling**).
- `chat`: Dispatches the API call with support for stop sequences and system prompts.

In [ ]:
# Load environment variables and create the Anthropic client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-3-5-haiku-latest"

In [ ]:
# Conversation helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Step 2: Synthetic Dataset Generation with Prefilling

### Why Synthetic Datasets?
When starting a new project, you rarely have 1,000 real production queries. You can use a fast model (like Claude Haiku) to synthetically generate realistic test cases.

### The Assistant Prefill Pattern (`"```json"`)
Claude allows pre-populating the start of the assistant's turn. When we prefill `"```json"` and add `stop_sequences=["```"]`:
1. Claude is forced to immediately start outputting JSON without preamble like *"Sure! Here are your items:"*.
2. Claude automatically stops when the closing code fence is generated.
3. `json.loads(text)` succeeds reliably without fragile regex stripping.

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Generated and saved {len(dataset)} items to dataset.json:")
print(json.dumps(dataset, indent=2))

## Step 3: Model-Based Grading (LLM-as-a-Judge)

### ⚠️ The "Middling Score" Trap
If you ask an LLM: *"Rate this output on a scale of 1 to 10"*, it will almost always reply with **6 or 7**. It hedges because it assumes *"it's not terrible, but could be better."*

### The Solution: Multi-Attribute Anchoring
We force the evaluator to systematically critique the output **before** producing a numerical score:
1. `strengths`: 1 to 3 concrete positive aspects.
2. `weaknesses`: 1 to 3 flaws or omissions.
3. `reasoning`: A synthesized evaluation rationale.
4. `score`: The calibrated numerical rating (1–10).

### XML Tag Prompt Structure
We wrap inputs in distinct XML tags (`<task>`, `<solution>`, `<criteria>`) so Claude distinguishes instructions from user content.

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Step 4: Candidate Prompt Execution (`run_prompt`)

This function represents the **system under test**.
We inject the test case's `task` into our prompt template. We use `"```code"` prefilling and `stop_sequences=["```"]` to instruct Claude to output raw code without chatty explanations.

In [ ]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

## Step 5: Deterministic Code-Based Syntax Validation

Why waste tokens asking an LLM if a JSON string has a missing comma? Deterministic code is:
1. **Instantaneous** (sub-millisecond).
2. **Zero cost** ($0.00).
3. **100% deterministic** (never hallucinates).

We implement three runtime validators:
- `validate_json`: Checks validity via `json.loads`
- `validate_python`: Compiles code into an Abstract Syntax Tree via `ast.parse`
- `validate_regex`: Compiles regex pattern via `re.compile`

Returns `10` for valid syntax, `0` for syntax errors.

In [ ]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

## Step 6: Test Case Execution & Composite Scoring

We tie the candidate prompt, model grader, and syntax grader together.
We compute a **composite score**:
$$\text{Score} = \frac{\text{Model Score} + \text{Syntax Score}}{2}$$

If the code looks plausible to an LLM but has a syntax error, the syntax score of `0` drops the overall score dramatically.

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

## Step 7: Batch Evaluation Suite & Summary Metrics

`run_eval(dataset)` iterates over every test case, runs the composite evaluation pipeline, and calculates the overall benchmark score across the test suite.

In [ ]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Step 8: Run Evaluation on Golden Test Dataset

In [ ]:
# Load persistent test cases and execute evaluation
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
# Pretty print complete structured results
print(json.dumps(results, indent=2))